In [1]:
import numpy as np
import pandas as pd
import PF_wrapper as PF
import time
import os

from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

In [2]:
rootdir ='/home/ben/Documents/pfgap/UCRArchive_2018'

thenames = []

for subdir, dirs, files in os.walk(rootdir):
    thename = subdir.replace(rootdir + '/','')
    if 'Missing' not in thename:
        thenames.extend([thename]) #break   

thenames = thenames[1:]

In [3]:
len(thenames)

128

In [4]:
# for each dataset:
    #1. train PF models for each representative distance in "enabled distances": PFdtw, PFerp, etc.
    #2. train a PF model using shapeHoG1dDTW
    #3. train a PF model using shifazDTW
    #4. train a PF model using the default distances: combination of distances for 1.
    #5. (previous steps using default parameters + output_directory=dataset_name, entry_separator="\t")
    #6. compute the accuracy and F1 scores for each approach (12 approaches)
    #7. repeat this 10 times, storing a list of pd dataframes.

In [13]:
def runPFGAPexperiment():
    #mydict = {}
    names = ["GunPoint"] #, "ItalyPowerDemand", "ArrowHead"]
    distances = ["dtw", "ddtw", "twe", "wdtw", "wddtw",
                "euclidean", "lcss", "msm", "erp", "shifazDTW", "shapeHoG1dDTW", "default"] # default means the first 9.
    
    columns = ["DTW", "DDTW", "TWE", "WDTW", "WDDTW",
                "Euclidean", "LCSS", "MSM", "ERP", "ShifazDTW", "ShapeDTW", "Default"]
    
    BigAccsDataFrame = pd.DataFrame(columns=columns) #, index=thenames) # the entries will be mean +/- std.
    BigF1DataFrame = pd.DataFrame(columns=columns) #, index=thenames) # the entries will be mean +/- std.
    
    #for name in names: # for testing.
    for name in thenames:
        filename = rootdir + '/' + name + '/' + name + "_TRAIN.tsv"
        filename_test = rootdir + '/' + name + '/' + name + "_TEST.tsv"
        df_train = pd.read_csv(filename, sep='\t', header=None)
        df_test = pd.read_csv(filename_test, sep='\t', header=None)
        
        if df_train.isnull().values.any():
            print(name + " contains nans: skipping")
            continue

        if df_test.isnull().values.any():
            print(name + " contains nans: skipping")
            continue

        #y_train = df_train.iloc[:, 0].to_numpy()
        y_test = df_test.iloc[:, 0].to_numpy()


        dataset_timeout = False
        run_timeout = False
        Acc = []
        F1 = []
        t0 = time.time() # we can give each dataset up to two hours. That's 10 minutes per distance configuration (1 minute per run).
        for dist in distances:
            t1 = time.time()
            if t1-t0 > 600*6:
                dataset_timeout = True
                continue

            if run_timeout:
                continue
            
            accs = []
            f1s = []
            
            for i in range(10):
                t2 = time.time()
                
                if t2-t1 > 60*6:
                    run_timeout = True
                    continue
                    
                if dist == "default":
                    modelname = name + str(i)
                    distancelist = None
                else:
                    modelname = name + dist + str(i)
                    distancelist = [dist]
                
                PF.train(filename, test_file=filename_test, model_name=modelname, 
                     output_directory=name, entry_separator="\t", distances=distancelist,
                        parallel_train=True)

                _f0 = open(name + "/Validation_Predictions.txt")
                _f1 = _f0.read()
                preds = eval("np.array(" + _f1 + ")")
                _f0.close()

                try:
                    acc = accuracy_score(y_test,preds)
                    accs.extend([acc])
                    f1 = f1_score(y_test,preds)
                    f1s.extend([f1])
                except:
                    run_timeout = True
                    continue
            
            #now that we've computed all of the accuracies and f1 scores for each run, we aggregate them.
            accs = np.array(accs)
            f1s = np.array(accs)

            # expressing the means as percentages will save space in the table.
            accs_mean = np.round(100*np.mean(accs),2)
            accs_std = np.round(100*np.std(accs),2)
            f1s_mean = np.round(100*np.mean(f1s),2)
            f1s_std = np.round(100*np.std(f1s),2)
            
            Acc.extend([str(accs_mean) + "±" + str(accs_std)])
            F1.extend([str(f1s_mean) + "±" + str(f1s_std)])
        
        #now that we've computed the mean accuracy and F1 scores for each distance, we fill out another row of a table
        if ((not dataset_timeout) and (not run_timeout)):
            BigAccsDataFrame.loc[name] = Acc
            BigF1DataFrame.loc[name] = F1
        #otherwise, we don't add these as rows, since we want to skip them
    # At this point, we're done making the dataframe tables.
    # now we're going to export the dataframes as .csv files for later use in latex.
    BigAccsDataFrame.to_csv("Accuracy.csv")
    BigF1DataFrame.to_csv("F1.csv")
        
    return

In [14]:
runPFGAPexperiment()


0:3mb
finished in 0:0:0.073

0:6mb
finished in 0:0:0.026

-----------------Repetition No: 1 (BirdChicken_TRAIN.tsv)   -----------------
Using: 3 MB, Free: 27 MB, Allocated Pool: 30 MB, Max Available: 1024 MB
core.ProximityForestResult@7cd84586
6.7.10.3.0.1.2.5.8.4.9.
Using: 11 MB, Free: 19 MB, Allocated Pool: 30 MB, Max Available: 1024 MB
*

0:4mb
finished in 0:0:0.056

0:6mb
finished in 0:0:0.027

-----------------Repetition No: 1 (BirdChicken_TRAIN.tsv)   -----------------
Using: 3 MB, Free: 27 MB, Allocated Pool: 30 MB, Max Available: 1024 MB
core.ProximityForestResult@7cd84586
5.4.7.9.2.8.1.3.0.6.10.
Using: 10 MB, Free: 20 MB, Allocated Pool: 30 MB, Max Available: 1024 MB
*

0:3mb
finished in 0:0:0.054

0:6mb
finished in 0:0:0.029

-----------------Repetition No: 1 (BirdChicken_TRAIN.tsv)   -----------------
Using: 3 MB, Free: 27 MB, Allocated Pool: 30 MB, Max Available: 1024 MB
core.ProximityForestResult@7cd84586
2.4.8.5.6.10.3.7.9.1.0.
Using: 13 MB, Free: 17 MB, Allocated Pool: 

In [ ]:
pd.read_csv("GunPoint/Validation_Predictions